# PoseFrame Studio on a Colab T4
Choose **Runtime → Change runtime type → T4 GPU**, connect to the hosted runtime, then run the cells in order. Your browser displays the editor; Colab runs YOLO.

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a hosted GPU runtime, reconnect, and rerun this cell."
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)

This cell sets up the bundled PoseFrame app. Choose your image ZIP in the editor after running the next cell.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
from base64 import b64decode
import os
import sys

root = Path("/content/poseframe")
root.mkdir(exist_ok=True)
blob = b64decode("...")
with ZipFile(BytesIO(blob)) as bundle:
    for name in ["app.py", "pose_inference.py", "index.html", "static/app.js", "static/style.css", "requirements.txt"]:
        target = root / name
        target.parent.mkdir(exist_ok=True)
        target.write_bytes(bundle.read(name))
os.chdir(root)
sys.path.insert(0, str(root))
%pip install -q -r requirements.txt "ultralytics>=8.4.0"

Mount Drive to keep annotations across sessions. Once the editor opens, choose **Open folder → Image ZIP** and select your ZIP of frames.

In [ ]:
from google.colab import drive, output
from werkzeug.serving import make_server
from threading import Thread
import requests

drive.mount("/content/drive")
os.environ["POSEFRAME_DATA"] = "/content/drive/MyDrive/PoseFrameStudio/data"
from app import app
if "poseframe_server" not in globals():
    poseframe_server = make_server("127.0.0.1", 0, app, threaded=True)
    Thread(target=poseframe_server.serve_forever, daemon=True).start()
port = poseframe_server.server_port
info = requests.get(f"http://127.0.0.1:{port}/api/devices", timeout=30).json()
assert info.get("cuda_available"), info
print("PoseFrame GPU:", info["gpu_name"])
output.serve_kernel_port_as_iframe(port, height=850)